# Chunk Size, Overlap, and Retrieval Thresholds — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/rag-systems/5-2/chunking-and-retrieval-thresholds.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/rag-systems/5-2/chunking-and-retrieval-thresholds.ipynb)

Companion to: [Chunk Size, Overlap, and Retrieval Thresholds in RAG Systems](https://anacodicai.org/learn/rag-systems/5-2/chunking-and-retrieval-thresholds)


## What you'll build

1. A **chunker** that splits a document into overlapping, word-count windows.
2. A **chunk-count formula** and a check that it actually predicts what the chunker does.
3. Two different **similarity scorers** (cosine, Jaccard) run over the identical chunks —
   to see with your own eyes that a "relevance threshold" is a property of the scorer,
   not a universal number.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
import re
import math
from collections import Counter

print("ready — standard library only")

ready — standard library only


## Step 1 — the document

A short, plain-text document we'll chunk. Try swapping in your own paragraph later
and re-running everything below — the point of this notebook is that the numbers
should move when you change the input, not stay fixed.

In [2]:
DOC = """
Setting up the deployment pipeline. Every service in the platform ships through
a three-stage pipeline: build, stage, and release. The build stage compiles the
service and runs the unit test suite. A build that fails any test is rejected
before it reaches the stage environment. The stage environment mirrors
production traffic patterns at ten percent scale, and every deploy sits there
for a minimum soak time before promotion is allowed.

Rolling back a release. If an error budget burns down faster than the alerting
threshold allows, the on-call engineer can trigger an automatic rollback to the
last known-good release. Rollbacks restore the previous container image and
the previous configuration bundle together, since restoring only one of the two
has caused mismatched-schema incidents in the past. A rollback does not restore
database migrations; those are forward-only and must be reverted by a separate,
manually reviewed migration.

Configuring the service mesh. The mesh routes traffic between services using
weighted routing rules stored in the control plane. A canary release starts at
five percent of traffic and increases in fixed steps only if the error rate and
p99 latency both stay under their configured thresholds for the full observation
window. If either metric breaches its threshold at any point in the window, the
canary is aborted and traffic reverts to one hundred percent on the previous
version.

Handling secrets. Secrets are never stored in the service repository. Each
service reads its secrets from a mounted volume populated at container start
by the secrets manager, and the volume is backed by tmpfs so secrets never
touch disk. Rotating a secret requires a restart of every pod that mounted it,
which the deployment controller schedules automatically within a maintenance
window.
""".strip()

total_words = len(DOC.split())
print(f"Document length: {total_words} words")

Document length: 285 words


## Step 2 — the chunker

A sliding window over the words, with overlap so a sentence that straddles a
boundary still appears whole in at least one chunk.

In [3]:
def chunk_text(text, chunk_size_words, overlap_words):
    words = text.split()
    if overlap_words >= chunk_size_words:
        raise ValueError("overlap must be smaller than chunk size")
    chunks, start = [], 0
    step = chunk_size_words - overlap_words
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size_words]))
        if start + chunk_size_words >= len(words):
            break
        start += step
    return chunks

chunks_60_15 = chunk_text(DOC, chunk_size_words=60, overlap_words=15)
print(f"chunk_size=60 overlap=15 -> {len(chunks_60_15)} chunks")
for i, c in enumerate(chunks_60_15):
    print(f"  chunk {i}: {len(c.split())} words")

chunk_size=60 overlap=15 -> 6 chunks
  chunk 0: 60 words
  chunk 1: 60 words
  chunk 2: 60 words
  chunk 3: 60 words
  chunk 4: 60 words
  chunk 5: 60 words


## Step 3 — does the formula predict the chunker?

```
step        = chunk_size - overlap
num_chunks  = ceil( max(total_words - overlap, 0) / step )
```

**Try it:** change `chunk_size_words` and `overlap_words` below and re-run — the
assertion will fail loudly if the formula and the real chunker ever disagree.

In [4]:
def expected_chunk_count(total_words, chunk_size_words, overlap_words):
    step = chunk_size_words - overlap_words
    return math.ceil(max(total_words - overlap_words, 0) / step)

# <-- change these two numbers and re-run
chunk_size_words = 30
overlap_words = 15

chunks = chunk_text(DOC, chunk_size_words, overlap_words)
predicted = expected_chunk_count(total_words, chunk_size_words, overlap_words)
print(f"chunk_size={chunk_size_words} overlap={overlap_words} -> "
      f"{len(chunks)} chunks (formula predicted {predicted})")
assert len(chunks) == predicted, "formula and chunker disagree -- investigate!"
print("formula matches the real chunker ✓")

chunk_size=30 overlap=15 -> 18 chunks (formula predicted 18)
formula matches the real chunker ✓


## Step 4 — two scorers, the same chunks, the same query

This is the core experiment. `cosine` and `jaccard` are two entirely reasonable,
simple similarity functions. Run them over the *identical* chunks against the
*identical* query and look at the score ranges.

In [5]:
def term_frequency_vector(text):
    return Counter(re.findall(r"[a-z]+", text.lower()))

def cosine_similarity(vec_a, vec_b):
    common = set(vec_a) & set(vec_b)
    dot = sum(vec_a[w] * vec_b[w] for w in common)
    mag_a = math.sqrt(sum(v * v for v in vec_a.values()))
    mag_b = math.sqrt(sum(v * v for v in vec_b.values()))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

def jaccard_similarity(vec_a, vec_b):
    set_a, set_b = set(vec_a), set(vec_b)
    return len(set_a & set_b) / len(set_a | set_b) if set_a and set_b else 0.0

query = "how does an automatic rollback work and what does it restore"
chunks = chunk_text(DOC, 60, 15)
q_vec = term_frequency_vector(query)

print(f"Query: {query}\n")
print(f"{'chunk':<7}{'cosine':<10}{'jaccard':<10}")
cos_scores, jac_scores = [], []
for i, c in enumerate(chunks):
    c_vec = term_frequency_vector(c)
    cos, jac = cosine_similarity(q_vec, c_vec), jaccard_similarity(q_vec, c_vec)
    cos_scores.append(cos)
    jac_scores.append(jac)
    print(f"{i:<7}{cos:<10.3f}{jac:<10.3f}")

print(f"\ncosine  range: {min(cos_scores):.3f} - {max(cos_scores):.3f}")
print(f"jaccard range: {min(jac_scores):.3f} - {max(jac_scores):.3f}")

Query: how does an automatic rollback work and what does it restore

chunk  cosine    jaccard   
0      0.092     0.041     
1      0.177     0.083     
2      0.256     0.109     
3      0.086     0.017     
4      0.054     0.018     
5      0.052     0.038     

cosine  range: 0.052 - 0.256
jaccard range: 0.017 - 0.109


## Step 5 — the takeaway, stated as a number

The two scorers agree on *which* chunk is the best match. They completely
disagree on *what number* counts as "confidently relevant" — which is exactly
why a threshold tuned for one scoring function stops meaning anything the
moment you swap it for a different embedding model or metric.

In [6]:
best_cosine = cos_scores.index(max(cos_scores))
best_jaccard = jac_scores.index(max(jac_scores))
assert best_cosine == best_jaccard, "the two scorers should agree on the winner"
assert "rollback" in chunks[best_cosine].lower()

cosine_range = max(cos_scores) - min(cos_scores)
jaccard_range = max(jac_scores) - min(jac_scores)
print(f"Both scorers picked chunk {best_cosine} as the best match (contains "
      f"'rollback': {'rollback' in chunks[best_cosine].lower()})")
print(f"cosine range width:  {cosine_range:.3f}")
print(f"jaccard range width: {jaccard_range:.3f}")
print(f"ratio: cosine's range is {cosine_range / jaccard_range:.2f}x wider than "
      f"jaccard's, on the SAME chunks, the SAME query")
print("\nA single hard-coded threshold (say, 0.15) would behave completely "
      "differently depending on which of these two scorers is live.")

Both scorers picked chunk 2 as the best match (contains 'rollback': True)
cosine range width:  0.204
jaccard range width: 0.092
ratio: cosine's range is 2.22x wider than jaccard's, on the SAME chunks, the SAME query

A single hard-coded threshold (say, 0.15) would behave completely differently depending on which of these two scorers is live.


## Try it yourself

- Swap `DOC` in Step 1 for your own text and re-run everything.
- In Step 3, try `chunk_size_words=200, overlap_words=20` — a much lower overlap
  fraction — and see how much the chunk count changes.
- In Step 4, write a query that's *barely* related to the document and watch
  both score ranges collapse toward zero — that's what "genuinely out of scope"
  looks like numerically, which is the setup for the abstention-gate notebook.